In [ ]:
#| export machine_learning.note_linking
import ast
import copy
from pathlib import Path
from os import PathLike

from trouver.obsidian.links import links_from_text, ObsidianLink
from trouver.machine_learning.note_data import NoteLinkEnum

from trouver.personal_vault.reference import index_note_for_reference
from trouver.obsidian.vault import VaultNote


In [ ]:
from unittest.mock import patch as mock_patch
from fastcore.test import *

### Link cache note

Using the model will take a lot of time --- not only does each prediction take about a few seconds, but also the predictions need to be made on pairs of notes and hence the total time needed for predictions grows quadratically with the number of notes. As such, "link cache notes" will be made to record predictions.

The link cache note will be saved in the root directory of its reference folder.


In [ ]:
#| export machine_learning.note_linking
def link_cache_note(
        vault: PathLike,
        reference: str,
        create_if_does_not_exist: bool = True,
        ) -> VaultNote: # The `VaultNote` object representing the link cache note.
    """
    Return a `VaultNote` object representing the link cache note in a reference of a vault.
    """
    ind_note: VaultNote = index_note_for_reference(vault, reference, update_cache=True)
    reference_folder: Path = ind_note.path(relative=True).parent
    vn = VaultNote(vault, rel_path=reference_folder / f'_link_cache_{reference}.md')
    if create_if_does_not_exist and not vn.exists():
        vn.create()
    return vn

The link cache note will be formatted as follows:

```
- [[origin_note_name_1]]
    - [[relied_note_name_1]]: [<comma_separated_link_types_1>]
    - [[relied_note_name_2]]: [<comma_separated_link_types_2>]
    ...
<blank space for separation>
- [[origin_note_name_2]]
    - ...
```

In [ ]:
#| export machine_learning.note_linking
def separate_blocks(
        text: str) -> list[str]:
    """
    Splits text into blocks separated by one or more blank lines.
    Returns a list of blocks (strings) with whitespace stripped.
    """
    blocks = []
    current_block = []
    
    for line in text.splitlines():
        if line.strip() == '':  # Blank line
            if current_block:  # Only add if we have content
                blocks.append('\n'.join(current_block))
                current_block = []
        else:
            current_block.append(line)
    
    # Add the last block if there's content remaining
    if current_block:
        blocks.append('\n'.join(current_block))
    
    return blocks

In [ ]:
text = """First line
Second line

Third block starts here
With multiple lines

Final block"""

blocks = separate_blocks(text)
print(blocks)
# for i, block in enumerate(blocks, 1):
#     print(f"Block {i}:\n{block}\n{'-'*20}")

['First line\nSecond line', 'Third block starts here\nWith multiple lines', 'Final block']


In [ ]:
#| export machine_learning.note_linking
def parse_link_cache_note(
        link_cache_note: VaultNote,
        ) -> dict[str, dict[str, list[NoteLinkEnum]]]: # The first key is the name of an "origin note". The second key is the name of a "relied note" with respect to the origin note. The value is a list of the link types from the origin note to the relied note.
    """
    See also `write_link_cache_note`, which is essentially the opposite of this function.
    """
    text = link_cache_note.text()
    blocks = separate_blocks(text)
    link_types: dict[str, dict[str, list[NoteLinkEnum]]] = {}
    for block in blocks:
        lines: list[str] = block.splitlines()
        first_line_link: ObsidianLink = links_from_text(lines[0])[0]
        origin_note_name = first_line_link.file_name
        link_types[origin_note_name] = {}
        for line in lines[1:]:
            link: ObsidianLink = links_from_text(line)[0]
            relied_note_name = link.file_name
            ind = line.index(':')
            note_type_list = ast.literal_eval(line[ind+2:])
            link_types[origin_note_name][relied_note_name] = [
                NoteLinkEnum[note_type_str] for note_type_str in note_type_list]
    return link_types

In [ ]:
with mock_patch('__main__.VaultNote') as mock_vault_note:
    mock_link_cache_note = mock_vault_note.return_value
    mock_link_cache_note.text.return_value = '''
- [[origin_note_1]]
    - [[relied_note_1]]: ['INFO_TO_INFO_IN_CONTENT', 'INFO_TO_INFO_VIA_NOTAT']
    - [[relied_note_2]]: ['INFO_TO_NOTAT_VIA_EMBEDDING']

- [[origin_note_2]]
    - [[relied_note_3]]: ['NOTAT_TO_NOTAT']
    - [[relied_note_4]]: ['NOTAT_TO_INFO', 'NOTAT_TO_INFO_VIA_NOTAT']
'''
    parse_link_cache_note(mock_link_cache_note)

In [ ]:
#| export machine_learning.note_linking
def write_link_cache_note(
        link_types: dict[str, dict[str, list[NoteLinkEnum]]],
        cache_note: VaultNote,
        ) -> None:
    """
    Overwrite the contents of the note represented by `link_cache_note` using the data
    from `link_types`.

    `link_cache_notes` is assumed to exist.

    See also `parse_link_cache_note`, which is essentially the opposite of this function.
    """
    chunks: list[str] = []
    for origin_note_name, relied_dict in link_types.items():
        chunk_text = f"- [[{origin_note_name}]]\n"
        for relied_note_name, link_type_list in relied_dict.items():
            chunk_text = f'{chunk_text}    - [[{relied_note_name}]]: {str([link_type.name for link_type in link_type_list])}\n'
        chunks.append(chunk_text)
    cache_note.write('\n\n'.join(chunks))

In [ ]:
with mock_patch('__main__.VaultNote') as mock_vault_note:
    mock_link_cache_note = mock_vault_note.return_value
    link_types = {
        'origin_note_1': {
            'relied_note_1': [
                NoteLinkEnum.INFO_TO_INFO_IN_CONTENT, NoteLinkEnum.INFO_TO_INFO_VIA_NOTAT],
            'relied_note_2': [
                NoteLinkEnum.INFO_TO_NOTAT_VIA_EMBEDDING]},
        'origin_note_2': {
            'relied_note_3': [
                NoteLinkEnum.NOTAT_TO_NOTAT],
            'relied_note_4': [
                NoteLinkEnum.NOTAT_TO_INFO, NoteLinkEnum.NOTAT_TO_INFO_VIA_NOTAT] }
    }
    write_link_cache_note(link_types, mock_link_cache_note)
    args, _ = mock_link_cache_note.write.call_args
    written_content = args[0]
    print(written_content)
    test_eq(
        written_content,
        '''- [[origin_note_1]]
    - [[relied_note_1]]: ['INFO_TO_INFO_IN_CONTENT', 'INFO_TO_INFO_VIA_NOTAT']
    - [[relied_note_2]]: ['INFO_TO_NOTAT_VIA_EMBEDDING']


- [[origin_note_2]]
    - [[relied_note_3]]: ['NOTAT_TO_NOTAT']
    - [[relied_note_4]]: ['NOTAT_TO_INFO', 'NOTAT_TO_INFO_VIA_NOTAT']
'''
        )

- [[origin_note_1]]
    - [[relied_note_1]]: ['INFO_TO_INFO_IN_CONTENT', 'INFO_TO_INFO_VIA_NOTAT']
    - [[relied_note_2]]: ['INFO_TO_NOTAT_VIA_EMBEDDING']


- [[origin_note_2]]
    - [[relied_note_3]]: ['NOTAT_TO_NOTAT']
    - [[relied_note_4]]: ['NOTAT_TO_INFO', 'NOTAT_TO_INFO_VIA_NOTAT']



In [ ]:
#| export machine_learning.note_linking
def consolidate_note_linking_predictions_into_cache(
        origin_note: VaultNote | str,
        predictions: dict[str, list[NoteLinkEnum]], # An output of `predict_note_linking``
        cache: dict[str, dict[str, list[NoteLinkEnum]]], # See `parse_link_cache_note`. The first key is the name of an "origin note". The second key is the name of a "relied note" with respect to the origin note. The value is a list of the link types from the origin note to the relied note.
        ):
    """
    Consolidate the outputs of `predict_note_linking` into a link cache.
    """
    if isinstance(origin_note, VaultNote):
        origin_note = origin_note.name
    if origin_note not in cache:
        cache[origin_note] = {}
    for relied_note_name, predicted_link_enums in predictions.items():
        if origin_note == relied_note_name:
            continue 
        predicted_link_enums = set(predicted_link_enums)
        predicted_link_enums = predicted_link_enums - {NoteLinkEnum.NO_LINK}
        if relied_note_name not in cache[origin_note]:
            cache[origin_note][relied_note_name] = []
        cached_link_enums = set(cache[origin_note][relied_note_name])
        link_enums = cached_link_enums | predicted_link_enums
        cache[origin_note][relied_note_name] = list(link_enums)

In [ ]:
predictions = {
    'relied_note_name_1': [NoteLinkEnum.INFO_TO_INFO_IN_CONTENT, NoteLinkEnum.INFO_TO_INFO_VIA_NOTAT],
    'relied_note_name_2': [],
    'relied_note_name_3': [NoteLinkEnum.INFO_TO_NOTAT_VIA_EMBEDDING]}
cache = {'origin_note_name': {'relied_note_name_1': [NoteLinkEnum.INFO_TO_INFO_IN_CONTENT, NoteLinkEnum.INFO_TO_INFO_IN_SEE_ALSO]}}

consolidate_note_linking_predictions_into_cache('origin_note_name', predictions, cache)

print(cache)
test_eq(
    set(cache['origin_note_name']['relied_note_name_1']), 
    set([NoteLinkEnum.INFO_TO_INFO_IN_CONTENT, NoteLinkEnum.INFO_TO_INFO_VIA_NOTAT, NoteLinkEnum.INFO_TO_INFO_IN_SEE_ALSO]))

test_eq(
    set(cache['origin_note_name']['relied_note_name_2']), 
    set([]))

test_eq(
    set(cache['origin_note_name']['relied_note_name_3']), 
    set([NoteLinkEnum.INFO_TO_NOTAT_VIA_EMBEDDING]))

{'origin_note_name': {'relied_note_name_1': [<NoteLinkEnum.INFO_TO_INFO_VIA_NOTAT: 3>, <NoteLinkEnum.INFO_TO_INFO_IN_SEE_ALSO: 2>, <NoteLinkEnum.INFO_TO_INFO_IN_CONTENT: 1>], 'relied_note_name_2': [], 'relied_note_name_3': [<NoteLinkEnum.INFO_TO_NOTAT_VIA_EMBEDDING: 4>]}}


In [ ]:
#| export machine_learning.note_linking
def consolidate_caches(
        cache_1: dict[str, dict[str, list[NoteLinkEnum]]], # See `parse_link_cache_note`. The first key is the name of an "origin note". The second key is the name of a "relied note" with respect to the origin note. The value is a list of the link types from the origin note to the relied note.
        cache_2: dict[str, dict[str, list[NoteLinkEnum]]],
        ) -> dict[str, dict[str, list[NoteLinkEnum]]]:
    new_cache: dict[str, dict[str, list[NoteLinkEnum]]] = copy.deepcopy(cache_1)
    for origin_note_name, origin_note_dict in cache_2.items():
        consolidate_note_linking_predictions_into_cache(
            origin_note_name, origin_note_dict, new_cache)
    return new_cache

In [ ]:
# Create two caches with some overlapping data
cache_a = {
    'Note_A': {'Note_B': [NoteLinkEnum.INFO_TO_INFO_IN_CONTENT]}
}
cache_b = {
    'Note_A': {'Note_B': [NoteLinkEnum.INFO_TO_INFO_IN_SEE_ALSO]},
    'Note_C': {'Note_D': [NoteLinkEnum.NOTAT_TO_INFO]}
}

# Consolidate them
merged = consolidate_caches(cache_a, cache_b)

# Verify the merge happened
print(merged['Note_A']['Note_B']) 
# Output: [<NoteLinkEnum...CONTENT>, <NoteLinkEnum...SEE_ALSO>]

[<NoteLinkEnum.INFO_TO_INFO_IN_SEE_ALSO: 2>, <NoteLinkEnum.INFO_TO_INFO_IN_CONTENT: 1>]


In [ ]:
#| hide
from fastcore.test import *

# --- Basic Edge Cases ---
# Empty Inputs
test_eq(consolidate_caches({}, {}), {})

# Idempotency (Duplicates)
dup_cache = {'A': {'B': [NoteLinkEnum.INFO_TO_INFO_IN_CONTENT]}}
res = consolidate_caches(dup_cache, dup_cache)
test_eq(len(res['A']['B']), 1)

# --- Complex Scenario Tests ---
# Base setup for the main test scenario
cache_1 = {
    'origin_note_1': {
        'relied_note_1': [NoteLinkEnum.INFO_TO_INFO_IN_CONTENT],
    },
    'origin_note_2': { # Only exists in cache_1
        'relied_note_1': [NoteLinkEnum.INFO_TO_INFO_IN_CONTENT],
    },
    'origin_note_3': { # Exists in both, but relied_note_2 is unique to cache_1
        'relied_note_2': [NoteLinkEnum.NOTAT_TO_INFO],
    }
}
cache_2 = {
    'origin_note_1': { # Exists in both, relied_note_1 exists in both
        'relied_note_1': [NoteLinkEnum.INFO_TO_INFO_IN_SEE_ALSO]
    },
    'origin_note_3': { # Exists in both, but relied_note_1 is unique to cache_2
        'relied_note_1': [NoteLinkEnum.NOTAT_TO_INFO_VIA_NOTAT]
    }
}
new_cache = consolidate_caches(cache_1, cache_2)

# Case 1: Deep Merge (Union of Lists)
test_eq(
    set(new_cache['origin_note_1']['relied_note_1']), 
    {NoteLinkEnum.INFO_TO_INFO_IN_SEE_ALSO, NoteLinkEnum.INFO_TO_INFO_IN_CONTENT}
)

# Case 2: Preservation of Left-Only Data
test_eq(
    new_cache['origin_note_2']['relied_note_1'], [NoteLinkEnum.INFO_TO_INFO_IN_CONTENT]
)

# Case 3: Partial Merge (Left Unique Key in Shared Parent)
test_eq(
    new_cache['origin_note_3']['relied_note_2'], [NoteLinkEnum.NOTAT_TO_INFO]
)

# Case 4: Partial Merge (Right Unique Key in Shared Parent)
test_eq(
    new_cache['origin_note_3']['relied_note_1'], [NoteLinkEnum.NOTAT_TO_INFO_VIA_NOTAT]
)

# Case 6: Completely New Origin Key (Right-Only Top Level)
cache_new_origin = {'Z': {'Y': [NoteLinkEnum.NOTAT_TO_INFO]}}
res_new = consolidate_caches(cache_1, cache_new_origin)
test_eq(res_new['Z']['Y'], [NoteLinkEnum.NOTAT_TO_INFO])
test_eq(len(res_new), 4) # origin_1, origin_2, origin_3 + Z

# Case 7: Empty Inputs (Identity)
test_eq(consolidate_caches(cache_1, {}), cache_1)
test_eq(consolidate_caches({}, cache_2), cache_2)

In [ ]:
#| export machine_learning.note_linking
def remove_blank_or_no_link_data_from_cache(
        cache: dict[str, dict[str, list[NoteLinkEnum]]], # See `parse_link_cache_note`. The first key is the name of an "origin note". The second key is the name of a "relied note" with respect to the origin note. The value is a list of the link types from the origin note to the relied note.
        ) -> dict[str, dict[str, list[NoteLinkEnum]]]: # A new cache, with lists that are either blank or which only contain `NoteLinkEnum.NO_LINK` are removed and with blank dict values are also removed..
    new_cache: dict[str, dict[str, list[NoteLinkEnum]]] = {} 
    for origin_note_name, origin_dict in cache.items():
        cleaned_dict: dict[str, list[NoteLinkEnum]] = {}
        for relied_note_name, listy in origin_dict.items():
            if not listy or (len(set(listy)) == 1 and listy[0] == NoteLinkEnum.NO_LINK):
                continue
            cleaned_dict[relied_note_name] = listy
        if cleaned_dict:
            new_cache[origin_note_name] = cleaned_dict
    return new_cache

In [ ]:
cache = {
    'origin_note_1': {},
    'origin_note_2': {
        'relied_note_1': [],
        'relied_note_2': [NoteLinkEnum.NO_LINK] 
    },
    'origin_note_3': {
        'relied_note_1': [NoteLinkEnum.INFO_TO_INFO_IN_CONTENT, NoteLinkEnum.INFO_TO_INFO_IN_SEE_ALSO]
    }}
output = remove_blank_or_no_link_data_from_cache(cache)
test_eq(
    output, 
    {'origin_note_3':
     {'relied_note_1':
      [NoteLinkEnum.INFO_TO_INFO_IN_CONTENT, NoteLinkEnum.INFO_TO_INFO_IN_SEE_ALSO]}}
)

In [ ]:
#| export machine_learning.note_linking
def remove_nonexistent_note_names_from_cache(
        cache: dict[str, dict[str, list[NoteLinkEnum]]], # See `parse_link_cache_note`. The first key is the name of an "origin note". The second key is the name of a "relied note" with respect to the origin note. The value is a list of the link types from the origin note to the relied note.
        vault: PathLike
        ) -> dict[str, dict[str, list[NoteLinkEnum]]]:
    """
    Remove names of nonexistent notes in `cache`.
    """
    cache_copy = copy.deepcopy(cache)
    keys = cache_copy.keys()
    for origin_note_name in list(keys):
        origin_note = VaultNote(vault, name=origin_note_name)
        if not origin_note.exists():
            cache_copy.pop(origin_note_name)
    for origin_note_name, origin_dict in cache_copy.items():
        keys = origin_dict.keys()
        for relied_note_name in list(keys):
            relied_note = VaultNote(vault, name=relied_note_name)
            if not relied_note.exists():
                origin_dict.pop(relied_note_name)
    return cache_copy